In [2]:
import pandas as pd
from google import genai
from dotenv import load_dotenv
import os
import json
import time

# Limpieza inicial

Se tienen los datos crudos de los resultados de la encuesta, ahora se hará una limpieza inicial sobre ellos. 

In [2]:
df = pd.read_csv('../data/dataset_encuesta.csv', encoding='latin-1', sep=';')

df.head()

,Id,Start time,Completion time,Email,Name,¿Aceptas participar voluntariamente en esta encuesta?\n,¿Ha recibido alguna vez un SMS que le pareció sospechoso?,¿De quién decía ser el mensaje?,¿A qué categoría pertenece la estafa que intentaron hacerle?,"Escriba el texto del primer SMS sospechoso que recibió, tal como apareció en su teléfono.","Escriba el texto de un SMS legítimo que haya recibido recientemente, tal como apareció en su teléfono.\n \nEjemplos de mensajes que puede compartir:\n Notificación de transacción bancaria Promoción de T",¿De qué tipo es el mensaje legítimo?
0,1,24/03/2026 9:33,24/03/2026 9:35,anonymous,NaN,"Si, acepto participar",Si,Banco o entidad financiera,"Financiero (préstamos, inversiones, criptomon...","Se autorizó un cargo por Q8,500 en BAC. Si no ...",¡Felicidades! Has ganado un Smartphone 5G de T...,"Promoción o aviso de operadora (Tigo, Claro, M..."
1,2,24/03/2026 9:33,24/03/2026 9:57,anonymous,NaN,"Si, acepto participar",Si,Empresa de paquetería o envíos,"Paquetes y envíos (maletas, importación, entr...","Hemos intentado entregar tu paquete dos veces,...",Eres un cliente DIAMANTE! Claro Premia tu Fide...,"Promoción o aviso de operadora (Tigo, Claro, M..."
2,3,24/03/2026 18:25,24/03/2026 18:26,anonymous,NaN,"Si, acepto participar",Si,Empresa de paquetería o envíos,"Paquetes y envíos (maletas, importación, entr...",NaN,Confirmación de entrega de pedido,Confirmación de compra o entrega
3,4,24/03/2026 18:24,24/03/2026 18:29,anonymous,NaN,"Si, acepto participar",Si,Empresa de paquetería o envíos,"Paquetes y envíos (maletas, importación, entr...",Hola correo central!! te han enviado un paquet...,Confirmar pedido,Código de verificación (2FA)
4,5,24/03/2026 18:31,24/03/2026 18:36,anonymous,NaN,"Si, acepto participar",Si,Empresa de paquetería o envíos,"Paquetes y envíos (maletas, importación, entr...","Intentamos entregar tu paquete, pero no pudimo...",Bicredit: Te recordamos que tienes un cambio d...,"Notificación bancaria (movimiento, saldo, alerta)"


Dividir en mensajes maliciosos y legítmos y eliminar los NA

In [3]:
df_spam = df[['Escriba el texto del primer SMS sospechoso que recibió, tal como apareció en su teléfono.', '¿A qué categoría pertenece la estafa que intentaron hacerle?']]
df_ham = df[['Escriba el texto de un SMS legítimo que haya recibido recientemente, tal como apareció en su teléfono.\n \nEjemplos de mensajes que puede compartir:\n  Notificación de transacción bancaria Promoción de T', '¿De qué tipo es el mensaje legítimo?']]

df_spam = df_spam.dropna()
df_ham = df_ham.dropna()

Cambiar el nombre de la columna que indica la categoría del mensaje a uno más corto

In [4]:
df_spam = df_spam.rename(columns={'Escriba el texto del primer SMS sospechoso que recibió, tal como apareció en su teléfono.': 'Mensaje', '¿A qué categoría pertenece la estafa que intentaron hacerle?': 'Categoria'})
df_ham = df_ham.rename(columns={'Escriba el texto de un SMS legítimo que haya recibido recientemente, tal como apareció en su teléfono.\n \nEjemplos de mensajes que puede compartir:\n  Notificación de transacción bancaria Promoción de T': 'Mensaje', '¿De qué tipo es el mensaje legítimo?': 'Categoria'})

Crear la columna *tipo* la cuál va a indicar si el mensaje es malicioso o no

In [5]:
df_spam['tipo'] = 'Spam'
df_ham['tipo'] = 'Ham'

In [6]:
df_spam.head()

,Mensaje,Categoria,tipo
0,"Se autorizó un cargo por Q8,500 en BAC. Si no ...","Financiero (préstamos, inversiones, criptomon...",Spam
1,"Hemos intentado entregar tu paquete dos veces,...","Paquetes y envíos (maletas, importación, entr...",Spam
3,Hola correo central!! te han enviado un paquet...,"Paquetes y envíos (maletas, importación, entr...",Spam
4,"Intentamos entregar tu paquete, pero no pudimo...","Paquetes y envíos (maletas, importación, entr...",Spam
6,Guatex: Su paquete ha llegado al almacén pero ...,"Paquetes y envíos (maletas, importación, entr...",Spam


In [7]:
df_ham.head()

,Mensaje,Categoria,tipo
0,¡Felicidades! Has ganado un Smartphone 5G de T...,"Promoción o aviso de operadora (Tigo, Claro, M...",Ham
1,Eres un cliente DIAMANTE! Claro Premia tu Fide...,"Promoción o aviso de operadora (Tigo, Claro, M...",Ham
2,Confirmación de entrega de pedido,Confirmación de compra o entrega,Ham
3,Confirmar pedido,Código de verificación (2FA),Ham
4,Bicredit: Te recordamos que tienes un cambio d...,"Notificación bancaria (movimiento, saldo, alerta)",Ham


Unir la información obtenida.

In [8]:
df_final =  pd.concat([df_spam, df_ham], ignore_index=True)

In [9]:
df_final.head()

,Mensaje,Categoria,tipo
0,"Se autorizó un cargo por Q8,500 en BAC. Si no ...","Financiero (préstamos, inversiones, criptomon...",Spam
1,"Hemos intentado entregar tu paquete dos veces,...","Paquetes y envíos (maletas, importación, entr...",Spam
2,Hola correo central!! te han enviado un paquet...,"Paquetes y envíos (maletas, importación, entr...",Spam
3,"Intentamos entregar tu paquete, pero no pudimo...","Paquetes y envíos (maletas, importación, entr...",Spam
4,Guatex: Su paquete ha llegado al almacén pero ...,"Paquetes y envíos (maletas, importación, entr...",Spam


In [10]:
df_final.to_csv('../data/dataset_tranformed.csv')

Generar data sintética. 

In [11]:
api_key = os.environ.get('GEMINI_API_KEY')

In [68]:
client = genai.Client(api_key=api_key)

all_variations = []

for idx, row in df_final.iterrows():
    prompt = f"""
Eres un experto en ciberseguridad guatemalteco analizando SMS fraudulentos.

Dado este mensaje SMS original:
- Texto: "{row['Mensaje']}"
- Tipo: "{row['tipo']}"
- Categoría: "{row['Categoria']}"

Genera exactamente 8 variaciones sintéticas que:
- Mantengan la misma etiqueta ({row['tipo']}) y categoría ({row['Categoria']})
- Usen instituciones guatemaltecas reales (EMETRA, SAT, Banrural, BAM, IGSS, Tigo, Claro, Renap, Migración)
- Usen español guatemalteco natural (mezcla de vos/usted según contexto)

{"Para variaciones SPAM (fraudulentos): Variar nivel de urgencia (mayoría sutil), incluir URLs falsas (.icu, .xyz, .info, .tk, bit.ly), variar montos entre Q150 y Q5,000" if row['tipo'] == 'spam' else "Para variaciones HAM (legítimos): NO incluir URLs sospechosas, mantener tono formal e institucional, variar montos, fechas y detalles menores"}

Responde ÚNICAMENTE con un array JSON válido con exactamente 8 objetos, sin texto adicional:
[
  {{
    "texto": "texto completo del SMS",
    "categoria": "{row['Categoria']}",
    "tipo": "{row['tipo']}"
  }}
]
"""
    
    try:
        response = client.models.generate_content(
            model="gemini-3-pro-preview",  
            config={"response_mime_type": "application/json"},
            contents=[prompt]
        )
        
        variations = json.loads(response.text)
        
        if len(variations) != 8:
            print(f"Fila {idx}: generó {len(variations)} variaciones en lugar de 8")
        
        all_variations.extend(variations)
        
        time.sleep(1)  
        
    except Exception as e:
        print(f" Error en fila {idx}: {e}")
        continue


df_synthetic = pd.DataFrame(all_variations)
df_synthetic.to_csv('../data/dataset_sintetico.csv', index=False)

In [69]:
df_synthetic.shape

(480, 3)

Unir los resultados y guardarlos. 

In [70]:
dataset_smishing = pd.concat([df_final, df_synthetic])

dataset_smishing.to_csv('../data/dataset_smishing.csv')

Finalmente, acortar los nombres del campo categoría

In [71]:
dataset_smishing['Categoria'].unique()

<StringArray>
[            ' Financiero (préstamos, inversiones, criptomonedas, pólizas, cheques)',
                      ' Paquetes y envíos (maletas, importación, entrega de bienes)',
 'Servicios y trabajo (ofertas de empleo, visas, servicios no realizados, talleres)',
                                                                     ' No sé / Otra',
   'Compra o venta de productos (vehículos, joyería, muebles, inmuebles, repuestos)',
                              'Premios y recompensas (vacaciones, sorteos, regalos)',
                                 ' Estafa afectiva (romance, familiar en problemas)',
                          'Digital y educación (venta de tareas, links sospechosos)',
                            'Promoción o aviso de operadora (Tigo, Claro, Movistar)',
                                                  'Confirmación de compra o entrega',
                                                     ' Código de verificación (2FA)',
                                 'Notifi

In [ ]:
mapeo = {
    'Financiero (préstamos, inversiones, criptomonedas, pólizas, cheques)': 'financiero',
    ' Paquetes y envíos (maletas, importación, entrega de bienes)': 'paquetes_envios',
    'Servicios y trabajo (ofertas de empleo, visas, servicios no realizados, talleres)': 'servicios_trabajo',
    ' No sé / Otra': 'otro',
    'Compra o venta de productos (vehículos, joyería, muebles, inmuebles, repuestos)': 'compra_venta',
    'Premios y recompensas (vacaciones, sorteos, regalos)': 'premios',
    ' Estafa afectiva (romance, familiar en problemas)': 'estafa_afectiva',
    'Digital y educación (venta de tareas, links sospechosos)': 'digital',
    'Promoción o aviso de operadora (Tigo, Claro, Movistar)': 'promo_operadora',
    'Confirmación de compra o entrega': 'confirmacion_compra',
    ' Código de verificación (2FA)': 'codigo_verificacion',
    'Notificación bancaria (movimiento, saldo, alerta)': 'notificacion_bancaria',
    'Publicidad legítima de una empresa conocida': 'publicidad',
    'Otro': 'otro',
    'Aviso gubernamental (SAT, IGSS, Renap, Migración)': 'aviso_gubernamental'
}

dataset_smishing['Categoria'] = dataset_smishing['Categoria'].map(mapeo)

In [74]:
dataset_smishing.to_csv('../data/dataset_smishing.csv')

In [75]:
dataset_smishing['Categoria'].unique()

<StringArray>
[                    nan,       'paquetes_envios',     'servicios_trabajo',
                  'otro',          'compra_venta',               'premios',
       'estafa_afectiva',               'digital',       'promo_operadora',
   'confirmacion_compra', 'notificacion_bancaria',            'publicidad',
   'aviso_gubernamental']
Length: 13, dtype: str

## Whitelist

Ahora, se generara la whitelist con información obtenida mediante el modelo *Claude 4.5 Sonnet*

In [76]:
# ─────────────────────────────────────────────────────────────────────
# SECTOR 1: TELECOMUNICACIONES
# ─────────────────────────────────────────────────────────────────────
whitelist_telecomunicaciones = [
    {
        "institucion": "Tigo Guatemala",
        "categoria": "telecomunicaciones",
        "numero": "24280000",
        "numero_normalizado": "+50224280000",
        "tipo_numero": "fijo",
        "canal": "call_center",
        "descripcion": "Atención al cliente Tigo (desde cualquier teléfono)",
        "shortcode": None,
        "whatsapp": True,
        "fuente": "tigo.com.gt"
    },
    {
        "institucion": "Tigo Guatemala",
        "categoria": "telecomunicaciones",
        "numero": "*611",
        "numero_normalizado": None,
        "tipo_numero": "shortcode",
        "canal": "call_center",
        "descripcion": "Atención al cliente Tigo (desde línea Tigo)",
        "shortcode": "*611",
        "whatsapp": False,
        "fuente": "tigo.com.gt"
    },
    {
        "institucion": "Tigo Money",
        "categoria": "telecomunicaciones",
        "numero": "24280041",
        "numero_normalizado": "+50224280041",
        "tipo_numero": "fijo",
        "canal": "call_center",
        "descripcion": "Atención al cliente Tigo Money",
        "shortcode": None,
        "whatsapp": True,
        "fuente": "ayudagt.tigomoney.com"
    },
    {
        "institucion": "Tigo Money",
        "categoria": "telecomunicaciones",
        "numero": "*987",
        "numero_normalizado": None,
        "tipo_numero": "shortcode",
        "canal": "call_center",
        "descripcion": "Atención Tigo Money (desde celular Tigo)",
        "shortcode": "*987",
        "whatsapp": False,
        "fuente": "ayudagt.tigomoney.com"
    },
    {
        "institucion": "Claro Guatemala",
        "categoria": "telecomunicaciones",
        "numero": "147100",
        "numero_normalizado": None,
        "tipo_numero": "shortcode",
        "canal": "call_center",
        "descripcion": "Servicio al cliente Claro (desde Claro o línea fija)",
        "shortcode": "147100",
        "whatsapp": False,
        "fuente": "@claroguatemala Twitter oficial"
    },
    {
        "institucion": "Claro Guatemala",
        "categoria": "telecomunicaciones",
        "numero": "22056295",
        "numero_normalizado": "+50222056295",
        "tipo_numero": "fijo",
        "canal": "call_center",
        "descripcion": "Línea alterna atención al cliente Claro",
        "shortcode": None,
        "whatsapp": False,
        "fuente": "claro.com.gt"
    },
]

# ─────────────────────────────────────────────────────────────────────
# SECTOR 2: BANCA
# ─────────────────────────────────────────────────────────────────────
whitelist_bancos = [
    {
        "institucion": "Banco Industrial (BI)",
        "categoria": "banca",
        "numero": "1717",
        "numero_normalizado": None,
        "tipo_numero": "shortcode",
        "canal": "contact_center",
        "descripcion": "Contact Center Banco Industrial — 24/7",
        "shortcode": "1717",
        "whatsapp": False,
        "fuente": "corporacionbi.com/gt/bancoindustrial"
    },
    {
        "institucion": "BAM (Banco Agromercantil)",
        "categoria": "banca",
        "numero": "23386565",
        "numero_normalizado": "+50223386565",
        "tipo_numero": "fijo",
        "canal": "contact_center",
        "descripcion": "Contact Center BAM — atención general y tarjetas",
        "shortcode": None,
        "whatsapp": False,
        "fuente": "bam.com.gt"
    },
    {
        "institucion": "Banrural",
        "categoria": "banca",
        "numero": "1720",
        "numero_normalizado": None,
        "tipo_numero": "shortcode",
        "canal": "contact_center",
        "descripcion": "Centro de Contacto Banrural",
        "shortcode": "1720",
        "whatsapp": False,
        "fuente": "tarjetasbanrural.com"
    },
    {
        "institucion": "Banrural",
        "categoria": "banca",
        "numero": "23266810",
        "numero_normalizado": "+50223266810",
        "tipo_numero": "fijo",
        "canal": "contact_center",
        "descripcion": "Teléfono directo Banrural tarjetas",
        "shortcode": None,
        "whatsapp": False,
        "fuente": "tarjetasbanrural.com"
    },
    {
        "institucion": "Banrural",
        "categoria": "banca",
        "numero": "23083000",
        "numero_normalizado": "+50223083000",
        "tipo_numero": "fijo",
        "canal": "banca_virtual",
        "descripcion": "Asistencia banca virtual Banrural",
        "shortcode": None,
        "whatsapp": False,
        "fuente": "bancavirtual.banrural.com.gt"
    },
    {
        "institucion": "BAC Credomatic Guatemala",
        "categoria": "banca",
        "numero": "1800",
        "numero_normalizado": None,
        "tipo_numero": "shortcode",
        "canal": "contact_center",
        "descripcion": "Contact Center BAC Guatemala — 24/7 tarjetas y banca",
        "shortcode": "1800",
        "whatsapp": False,
        "fuente": "baccredomatic.com/es-gt"
    },
    {
        "institucion": "BAC Credomatic Guatemala",
        "categoria": "banca",
        "numero": "22782000",
        "numero_normalizado": "+50222782000",
        "tipo_numero": "fijo",
        "canal": "contact_center",
        "descripcion": "Teléfono directo BAC Guatemala",
        "shortcode": None,
        "whatsapp": False,
        "fuente": "baccredomatic.com/es-gt"
    },
    {
        "institucion": "G&T Continental",
        "categoria": "banca",
        "numero": "1500",
        "numero_normalizado": None,
        "tipo_numero": "shortcode",
        "canal": "contact_center",
        "descripcion": "Contact Center G&T Continental",
        "shortcode": "1500",
        "whatsapp": False,
        "fuente": "gtcontinental.com.gt"
    },
    {
        "institucion": "G&T Continental",
        "categoria": "banca",
        "numero": "24101500",
        "numero_normalizado": "+50224101500",
        "tipo_numero": "fijo",
        "canal": "contact_center",
        "descripcion": "Teléfono directo G&T Continental",
        "shortcode": None,
        "whatsapp": False,
        "fuente": "gtcontinental.com.gt"
    },
    {
        "institucion": "Banco de los Trabajadores (Bantrab)",
        "categoria": "banca",
        "numero": "1503",
        "numero_normalizado": None,
        "tipo_numero": "shortcode",
        "canal": "contact_center",
        "descripcion": "Contact Center Bantrab",
        "shortcode": "1503",
        "whatsapp": False,
        "fuente": "bantrab.com.gt"
    },
    {
        "institucion": "Banco de los Trabajadores (Bantrab)",
        "categoria": "banca",
        "numero": "24101503",
        "numero_normalizado": "+50224101503",
        "tipo_numero": "fijo",
        "canal": "contact_center",
        "descripcion": "Teléfono directo Bantrab",
        "shortcode": None,
        "whatsapp": False,
        "fuente": "bantrab.com.gt"
    },
    {
        "institucion": "Vivibanco (Banco Vivibanco)",
        "categoria": "banca",
        "numero": "23690000",
        "numero_normalizado": "+50223690000",
        "tipo_numero": "fijo",
        "canal": "contact_center",
        "descripcion": "Atención al cliente Vivibanco Guatemala",
        "shortcode": None,
        "whatsapp": False,
        "fuente": "vivibanco.com.gt"
    },
    {
        "institucion": "Citibank Guatemala",
        "categoria": "banca",
        "numero": "18002288888",
        "numero_normalizado": None,
        "tipo_numero": "toll_free",
        "canal": "contact_center",
        "descripcion": "Línea gratuita Citibank Guatemala",
        "shortcode": None,
        "whatsapp": False,
        "fuente": "citibank.com.gt"
    },
]

# ─────────────────────────────────────────────────────────────────────
# SECTOR 3: GOBIERNO
# ─────────────────────────────────────────────────────────────────────
whitelist_gobierno = [
    {
        "institucion": "SAT (Superintendencia de Administración Tributaria)",
        "categoria": "gobierno",
        "numero": "1550",
        "numero_normalizado": None,
        "tipo_numero": "shortcode",
        "canal": "call_center",
        "descripcion": "Línea SAT para atención al contribuyente",
        "shortcode": "1550",
        "whatsapp": False,
        "fuente": "@SATGT Twitter oficial"
    },
    {
        "institucion": "SAT (Superintendencia de Administración Tributaria)",
        "categoria": "gobierno",
        "numero": "22174000",
        "numero_normalizado": "+50222174000",
        "tipo_numero": "fijo",
        "canal": "call_center",
        "descripcion": "Teléfono directo SAT atención al contribuyente",
        "shortcode": None,
        "whatsapp": False,
        "fuente": "@SATGT Twitter oficial"
    },
    {
        "institucion": "IGSS (Instituto Guatemalteco de Seguridad Social)",
        "categoria": "gobierno",
        "numero": "1522",
        "numero_normalizado": None,
        "tipo_numero": "shortcode",
        "canal": "contact_center",
        "descripcion": "Centro de Contacto IGSS",
        "shortcode": "1522",
        "whatsapp": False,
        "fuente": "igssgt.org/contactenos"
    },
    {
        "institucion": "IGSS (Instituto Guatemalteco de Seguridad Social)",
        "categoria": "gobierno",
        "numero": "24121224",
        "numero_normalizado": "+50224121224",
        "tipo_numero": "fijo",
        "canal": "contact_center",
        "descripcion": "Teléfono directo IGSS",
        "shortcode": None,
        "whatsapp": False,
        "fuente": "igssgt.org/contactenos"
    },
    {
        "institucion": "RENAP (Registro Nacional de las Personas)",
        "categoria": "gobierno",
        "numero": "1516",
        "numero_normalizado": None,
        "tipo_numero": "shortcode",
        "canal": "call_center",
        "descripcion": "Call Center RENAP",
        "shortcode": "1516",
        "whatsapp": False,
        "fuente": "renap.gob.gt"
    },
    {
        "institucion": "RENAP (Registro Nacional de las Personas)",
        "categoria": "gobierno",
        "numero": "24161900",
        "numero_normalizado": "+50224161900",
        "tipo_numero": "fijo",
        "canal": "pbx",
        "descripcion": "PBX RENAP (opción 2 para Centro de Información)",
        "shortcode": None,
        "whatsapp": False,
        "fuente": "renap.gob.gt"
    },
    {
        "institucion": "RENAP (Registro Nacional de las Personas)",
        "categoria": "gobierno",
        "numero": "42107327",
        "numero_normalizado": "+50242107327",
        "tipo_numero": "movil",
        "canal": "whatsapp",
        "descripcion": "WhatsApp oficial RENAP",
        "shortcode": None,
        "whatsapp": True,
        "fuente": "renap.gob.gt"
    },
    {
        "institucion": "IGM (Instituto Guatemalteco de Migración)",
        "categoria": "gobierno",
        "numero": "24112411",
        "numero_normalizado": "+50224112411",
        "tipo_numero": "fijo",
        "canal": "central",
        "descripcion": "Línea central Instituto Guatemalteco de Migración",
        "shortcode": None,
        "whatsapp": False,
        "fuente": "igm.gob.gt"
    },
    {
        "institucion": "EMETRA (Empresa Metropolitana de Transporte)",
        "categoria": "gobierno",
        "numero": "1551",
        "numero_normalizado": None,
        "tipo_numero": "shortcode",
        "canal": "emergencias",
        "descripcion": "Emergencias viales y Policía Municipal de Tránsito",
        "shortcode": "1551",
        "whatsapp": False,
        "fuente": "mundochapin.com"
    },
    {
        "institucion": "EMETRA (Empresa Metropolitana de Transporte)",
        "categoria": "gobierno",
        "numero": "22858400",
        "numero_normalizado": "+50222858400",
        "tipo_numero": "fijo",
        "canal": "central",
        "descripcion": "Teléfono directo EMETRA",
        "shortcode": None,
        "whatsapp": False,
        "fuente": "mundochapin.com"
    },
]

# ─────────────────────────────────────────────────────────────────────
# SECTOR 4: PAQUETERÍA / COURIER
# ─────────────────────────────────────────────────────────────────────
whitelist_paqueteria = [
    {
        "institucion": "DHL Guatemala",
        "categoria": "paqueteria",
        "numero": "23901100",
        "numero_normalizado": "+50223901100",
        "tipo_numero": "fijo",
        "canal": "call_center",
        "descripcion": "Atención al cliente DHL Guatemala",
        "shortcode": None,
        "whatsapp": False,
        "fuente": "@dhlguatemala Twitter oficial"
    },
    {
        "institucion": "FedEx Guatemala",
        "categoria": "paqueteria",
        "numero": "24112100",
        "numero_normalizado": "+50224112100",
        "tipo_numero": "fijo",
        "canal": "call_center",
        "descripcion": "Servicio al cliente FedEx — desde cualquier teléfono en Guatemala",
        "shortcode": None,
        "whatsapp": False,
        "fuente": "fedex.com/es-gt/customer-support/call-us.html"
    },
    {
        "institucion": "FedEx Guatemala",
        "categoria": "paqueteria",
        "numero": "1801003339",
        "numero_normalizado": None,
        "tipo_numero": "toll_free",
        "canal": "call_center",
        "descripcion": "Línea gratuita FedEx Guatemala",
        "shortcode": "1801003339",
        "whatsapp": False,
        "fuente": "fedex.com/es-gt/customer-support/call-us.html"
    },
    {
        "institucion": "Guatex",
        "categoria": "paqueteria",
        "numero": "23230500",
        "numero_normalizado": "+50223230500",
        "tipo_numero": "fijo",
        "canal": "call_center",
        "descripcion": "Central Guatex — Zona 12, Ciudad de Guatemala",
        "shortcode": None,
        "whatsapp": False,
        "fuente": "latinoplaces.com (directorio verificado)"
    },
    {
        "institucion": "Cargo Expreso",
        "categoria": "paqueteria",
        "numero": "24744444",
        "numero_normalizado": "+50224744444",
        "tipo_numero": "fijo",
        "canal": "call_center",
        "descripcion": "Call center Cargo Expreso — rastreo y atención al cliente",
        "shortcode": None,
        "whatsapp": False,
        "fuente": "cargoexpreso.com / guaterastreo.com"
    },
]

# ─────────────────────────────────────────────────────────────────────
# SECTOR 5: RESTAURANTES Y DELIVERY
# ─────────────────────────────────────────────────────────────────────
whitelist_restaurantes = [
    {
        "institucion": "PedidosYa Guatemala",
        "categoria": "delivery_app",
        "numero": "36918935",
        "numero_normalizado": "+50236918935",
        "tipo_numero": "movil",
        "canal": "soporte_cliente",
        "descripcion": "Soporte PedidosYa Guatemala (ex-Hugo)",
        "shortcode": None,
        "whatsapp": True,
        "fuente": "numeroservicioalcliente.com — PedidosYa Guatemala"
    },
    {
        "institucion": "Pollo Campero Guatemala",
        "categoria": "restaurante_cadena_gt",
        "numero": "web_app_only",
        "numero_normalizado": None,
        "tipo_numero": "web_app_only",
        "canal": "app_web",
        "descripcion": "Pedidos solo vía app/web — cualquier SMS con número es sospechoso",
        "shortcode": None,
        "whatsapp": False,
        "fuente": "campero.com/gt"
    },
    {
        "institucion": "McDonald's Guatemala",
        "categoria": "restaurante_cadena_intl",
        "numero": "24154600",
        "numero_normalizado": "+50224154600",
        "tipo_numero": "fijo",
        "canal": "servicio_cliente",
        "descripcion": "Servicio al cliente McDonald's Guatemala (corporativo)",
        "shortcode": None,
        "whatsapp": False,
        "fuente": "Facebook oficial McDonald's Guatemala"
    },
    {
        "institucion": "McDonald's Guatemala",
        "categoria": "restaurante_cadena_intl",
        "numero": "44701770",
        "numero_normalizado": "+50244701770",
        "tipo_numero": "movil",
        "canal": "whatsapp_pedidos",
        "descripcion": "WhatsApp pedidos McDelivery Guatemala",
        "shortcode": None,
        "whatsapp": True,
        "fuente": "mcdonalds.com.gt/mcdelivery"
    },
    {
        "institucion": "McDonald's Guatemala",
        "categoria": "restaurante_cadena_intl",
        "numero": "23606363",
        "numero_normalizado": "+50223606363",
        "tipo_numero": "fijo",
        "canal": "contact_center_pedidos",
        "descripcion": "Teléfono McDelivery — pedidos y reclamos domicilio",
        "shortcode": None,
        "whatsapp": False,
        "fuente": "mcdonalds.com.gt/mcdelivery"
    },
    {
        "institucion": "Burger King Guatemala",
        "categoria": "restaurante_cadena_intl",
        "numero": "22000000",
        "numero_normalizado": "+50222000000",
        "tipo_numero": "fijo",
        "canal": "call_center_pedidos",
        "descripcion": "Call center y WhatsApp Burger King Guatemala",
        "shortcode": None,
        "whatsapp": True,
        "fuente": "info.bk.gt/llamanos"
    },
    {
        "institucion": "Pizza Hut Guatemala",
        "categoria": "restaurante_cadena_intl",
        "numero": "1744",
        "numero_normalizado": None,
        "tipo_numero": "shortcode",
        "canal": "call_center_pedidos",
        "descripcion": "Línea directa Pizza Hut domicilio Guatemala",
        "shortcode": "1744",
        "whatsapp": False,
        "fuente": "directorio.guatemala.com"
    },
]

df_telecom = pd.DataFrame(whitelist_telecomunicaciones)
df_bancos = pd.DataFrame(whitelist_bancos)
df_gobierno = pd.DataFrame(whitelist_gobierno)
df_paqueteria = pd.DataFrame(whitelist_paqueteria)
df_restaurantes = pd.DataFrame(whitelist_restaurantes)

df_whitelist = pd.concat(
    [
        df_telecom,
        df_bancos,
        df_gobierno,
        df_paqueteria,
        df_restaurantes
    ],
    ignore_index=True
)

df_whitelist.to_csv('../data/whitelist.csv')



In [4]:

whitelist_df = pd.read_csv("../data/whitelist.csv")

whitelist_df['numero']

0         24280000
1             *611
2         24280041
3             *987
4           147100
5         22056295
6             1717
7         23386565
8             1720
9         23266810
10        23083000
11            1800
12        22782000
13            1500
14        24101500
15            1503
16        24101503
17        23690000
18     18002288888
19            1550
20        22174000
21            1522
22        24121224
23            1516
24        24161900
25        42107327
26        24112411
27            1551
28        22858400
29        23901100
30        24112100
31      1801003339
32        23230500
33        24744444
34        36918935
35    web_app_only
36        24154600
37        44701770
38        23606363
39        22000000
40            1744
Name: numero, dtype: str